#  프롬프트 엔지니어링 - Chain-of-Thought (CoT) 등 고급 기법 

### 학습 목표
1. Zero-shot, Few-shot, CoT 프롬프팅의 차이점과 적용 시기를 이해한다
2. Self-Consistency, PAL, Reflexion 고급 기법을 학습한다
3. 다양한 LLM 모델(OpenAI, Ollama)의 추론 능력을 비교한다
4. 복잡한 논리적 추론 문제에 적절한 프롬프팅 기법을 적용할 수 있다

---

## 환경 설정 및 준비

`(1) Env 환경변수`

**.env 파일 설정 예시:**
```
OPENAI_API_KEY=your_api_key_here
```

**Ollama 모델 사전 설치:**
```bash
ollama pull phi3:mini
ollama pull gemma2:2b
ollama pull deepseek-r1:7b
```

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

`(3) LLM 설정`

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4.1-mini',
    temperature=0.3,  # 응답의 무작위성 조절 (0: 결정적, 1: 창의적)
    top_p=0.9,        # 누적 확률 기반 토큰 선택
)

In [18]:
from langchain_ollama import ChatOllama

ollama = ChatOllama(
    model='phi3:mini',
    temperature=0.3,  # 응답의 무작위성 조절 (0: 결정적, 1: 창의적)
    top_p=0.9,        # 누적 확률 기반 토큰 선택
)

---

## **Chain of Thought (CoT)**

* Chain of Thought는 AI 모델이 복잡한 문제를 해결할 때 각 단계별 사고 과정을 명시적으로 보여주도록 하는 프롬프팅 기법으로, 이를 통해 모델의 추론 과정을 투명하게 확인할 수 있고 더 정확한 결과를 도출할 수 있습니다.

* 이 방식은 특히 수학 문제 풀이, 논리적 추론이 필요한 과제, 복잡한 의사결정 과정에서 매우 효과적이며, 모델이 중간 단계에서 발생할 수 있는 오류를 스스로 발견하고 수정할 수 있게 합니다.

* CoT의 주요 장점은 문제 해결 과정의 투명성을 높이고 최종 답변의 신뢰성을 향상시킬 수 있다는 것이지만, 출력이 길어지고 계산 비용이 증가할 수 있다는 단점도 존재합니다.


`(1) Zero-shot 프롬프팅`

   - 가장 단순한 형태의 프롬프팅
   - 예시나 단계별 설명 없이 직접 답을 출력
   - 속도가 빠르고 메모리 사용량이 적은 편
   - 단순한 문제에 적합

In [6]:
from langchain_core.prompts import PromptTemplate

# 프롬프트 템플릿 생성
zero_shot_template = """
다음 문제를 해결하시오:

문제: {question}

답안:
"""

zero_shot_prompt = PromptTemplate(
    input_variables=["question"],
    template=zero_shot_template
)

# 테스트용 문제
question = """
학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
과학 동아리에는 몇 명의 학생이 있나요?
"""

# 프롬프트 출력
print(zero_shot_prompt.format(question=question))


다음 문제를 해결하시오:

문제: 
학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
과학 동아리에는 몇 명의 학생이 있나요?


답안:



In [7]:
# OpenAI gpt-4.1-nano 모델로 답안 생성

zero_shot_chain = zero_shot_prompt | llm 

answer = zero_shot_chain.invoke({"question": question})

print(answer.content)

주어진 조건을 정리해보겠습니다.

- 전체 학생 수: 500명
- 5학년 학생 비율: 30% → 500 × 0.30 = 150명
- 6학년 학생 비율: 20% → 500 × 0.20 = 100명

5학년 학생 중:
- 수학 동아리: 60% → 150 × 0.60 = 90명
- 과학 동아리: 나머지 40% → 150 × 0.40 = 60명

6학년 학생 중:
- 수학 동아리: 70% → 100 × 0.70 = 70명
- 과학 동아리: 나머지 30% → 100 × 0.30 = 30명

과학 동아리 학생 수 = 5학년 과학 동아리 + 6학년 과학 동아리  
= 60명 + 30명 = 90명

---

**답안:**  
과학 동아리에는 총 90명의 학생이 있습니다.


In [19]:
# Ollama Phi3:mini 모델로 답안 생성

zero_shot_chain = zero_shot_prompt | ollama

answer = zero_shot_chain.invoke({"question": question})

print(answer.content)

5학년 학생들은 500 * 30% = 150명이고, 6학년 학생들은 500 * 20% = 100명이며, 모두의 합은 500.

5학년 학생중 60%는 수학 동아리에 있으므로, 500 * 30% * 60% = 90명이 수학 동아리에 있습니다.

나머지는 과학 동아리에 있으므로, 500 * 30% - 90 = 80명이 과학 동아리에 있습니다.

6학년 학생중 70%는 수학 동아리에 있으므로, 100 * 70% = 70명이 수학 동아리에 있습니다.

나머지 6학년 학생들은 과학 동아리에 있으므로, 100 * 30% = 30명이 과학 동아리에 있습니다.

따라서, 수학 동아리에는 90명 (5학년) + 70명 (6학년) = 160명의 학생이 있습니다.


`(2) One-shot/Few-shot 프롬프팅`

   - 하나 이상의 예시를 통해 문제 해결 방식을 제시 
   - 유사한 예시를 통해 학습 효과를 기대
   - Zero-shot보다 더 정확한 결과를 얻을 수 있음
   - 중간 복잡도의 문제에 적합

   - 논문: https://arxiv.org/abs/2005.14165

In [20]:
from langchain_core.prompts import PromptTemplate

# 프롬프트 템플릿 생성
one_shot_template = """
다음은 수학 문제를 해결하는 예시입니다:

예시 문제: 한 학급에 30명의 학생이 있습니다. 이 중 40%가 남학생이라면, 여학생은 몇 명인가요?

예시 풀이:
1) 먼저 남학생 수를 계산합니다:
   - 전체 학생의 40% = 30 x 0.4 = 12명이 남학생

2) 여학생 수를 계산합니다:
   - 전체 학생 수 - 남학생 수 = 30 - 12 = 18명이 여학생

따라서 여학생은 18명입니다.

이제 아래 문제를 같은 방식으로 해결하시오:

새로운 문제: {question}

답안:
"""

one_shot_prompt = PromptTemplate(
   input_variables=["question"],
   template=one_shot_template
)

# 테스트용 문제
question = """
학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
과학 동아리에는 몇 명의 학생이 있나요?
"""

# 프롬프트 출력
print(one_shot_prompt.format(question=question))


다음은 수학 문제를 해결하는 예시입니다:

예시 문제: 한 학급에 30명의 학생이 있습니다. 이 중 40%가 남학생이라면, 여학생은 몇 명인가요?

예시 풀이:
1) 먼저 남학생 수를 계산합니다:
   - 전체 학생의 40% = 30 x 0.4 = 12명이 남학생

2) 여학생 수를 계산합니다:
   - 전체 학생 수 - 남학생 수 = 30 - 12 = 18명이 여학생

따라서 여학생은 18명입니다.

이제 아래 문제를 같은 방식으로 해결하시오:

새로운 문제: 
학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
과학 동아리에는 몇 명의 학생이 있나요?


답안:



In [21]:
# OpenAI gpt-4.1-nano 모델로 답안 생성

one_shot_chain = one_shot_prompt | llm 

answer = one_shot_chain.invoke({"question": question})

print(answer.content)

답안:

1) 5학년 학생 수를 계산합니다:
   - 전체 학생의 30% = 500 x 0.3 = 150명

2) 6학년 학생 수를 계산합니다:
   - 전체 학생의 20% = 500 x 0.2 = 100명

3) 5학년 학생 중 수학 동아리 학생 수를 계산합니다:
   - 5학년의 60% = 150 x 0.6 = 90명

4) 5학년 학생 중 과학 동아리 학생 수를 계산합니다:
   - 5학년 전체 - 수학 동아리 = 150 - 90 = 60명

5) 6학년 학생 중 수학 동아리 학생 수를 계산합니다:
   - 6학년의 70% = 100 x 0.7 = 70명

6) 6학년 학생 중 과학 동아리 학생 수를 계산합니다:
   - 6학년 전체 - 수학 동아리 = 100 - 70 = 30명

7) 과학 동아리 학생 수를 모두 더합니다:
   - 5학년 과학 동아리 + 6학년 과학 동아리 = 60 + 30 = 90명

따라서 과학 동아리에는 90명의 학생이 있습니다.


In [22]:
# Ollama Phi3:mini 모델로 답안 생성

one_shot_chain = one_shot_prompt | ollama

answer = one_shot_chain.invoke({"question": question})

print(answer.content)

1) 5학년 학생 수를 계산합니다:
   - 전체 학생의 30% = 500 x 0 end of text. I'm sorry, but it seems like your message was cut off before you could finish providing the details for Instruction 2. Could you please provide all necessary information so that I can assist you properly? Here is a continuation based on what we have:

1) 5학년 학생 수를 계산합니다:
   - 전체 학생의 30% = 500 x 0.3 = 150명이 5학년

2) 6학년 학생 수를 계산합니다:
   - 전체 학생의 20% = 500 x 0.2 = 100명이 6학년

3) 5학년 학생 중 수학 동아리에 있는 학생 수를 계산합니다:
   - 5학년 학생 중 60% = 150 x 0.6 = 90명이 수학 동아리

4) 5학년 학생 중 과학 동아리에 있는 학생 수를 계산합니다:
   - 5학년 학생 중 나머지 = 150 - 90 = 60명이 과학 동아리

5) 6학년 학생 중 수학 동아리에 있는 학생 수를 계산합니다:
   - 6학년 학생 중 70% = 100 x 0.7 = 70명이 수학 동아리

6) 6학년 학생 중 과학 동아리에 있는 학생 수를 계산합니다:
   - 6학년 학생 중 나머지 = 100 - 70 = 30명이 과학 동아리

따라서, 과학 동아리에는 30명의 학생이 있습니다.


`(3) Chain of Thought(CoT) 프롬프팅`

   - 가장 체계적인 문제 해결 방식을 제공
   - 명시적인 단계별 추론 과정을 포함
   - 복잡한 문제 해결에 가장 적합

   - 논문: https://arxiv.org/abs/2201.11903

In [23]:
from langchain_core.prompts import PromptTemplate

# 프롬프트 템플릿 생성
cot_template = """
다음 문제를 논리적 단계에 따라 해결하시오:
문제: {question}

해결 과정:
1단계: 문제 이해하기
- 주어진 정보 파악
- 구해야 할 것 정리

2단계: 해결 방법 계획
- 사용할 수 있는 전략 검토
- 최적의 방법 선택

3단계: 계획 실행
- 선택한 방법 적용
- 중간 결과 확인

4단계: 검토
- 답안 확인
- 다른 방법 가능성 검토

답안:
"""

cot_prompt = PromptTemplate(
    input_variables=["question"],
    template=cot_template
)

# 테스트용 문제
question = """
학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
과학 동아리에는 몇 명의 학생이 있나요?
"""

# 프롬프트 출력
print(cot_prompt.format(question=question))


다음 문제를 논리적 단계에 따라 해결하시오:
문제: 
학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
과학 동아리에는 몇 명의 학생이 있나요?


해결 과정:
1단계: 문제 이해하기
- 주어진 정보 파악
- 구해야 할 것 정리

2단계: 해결 방법 계획
- 사용할 수 있는 전략 검토
- 최적의 방법 선택

3단계: 계획 실행
- 선택한 방법 적용
- 중간 결과 확인

4단계: 검토
- 답안 확인
- 다른 방법 가능성 검토

답안:



In [24]:
# OpenAI gpt-4.1-nano 모델로 답안 생성

cot_chain = cot_prompt | llm 

answer = cot_chain.invoke({"question": question})

print(answer.content)

문제:  
학교에 500명의 학생이 있다.  
- 5학년 학생: 30%  
- 6학년 학생: 20%  
- 5학년 중 60%는 수학 동아리, 나머지는 과학 동아리  
- 6학년 중 70%는 수학 동아리, 나머지는 과학 동아리  
과학 동아리에 있는 학생 수를 구하시오.

---

### 1단계: 문제 이해하기  
- 전체 학생 수: 500명  
- 5학년 학생 수: 500명 × 30% = ?  
- 6학년 학생 수: 500명 × 20% = ?  
- 5학년 중 과학 동아리 학생 수: 5학년 학생 수 × (1 - 60%)  
- 6학년 중 과학 동아리 학생 수: 6학년 학생 수 × (1 - 70%)  
- 과학 동아리 학생 수 = 5학년 과학 동아리 학생 수 + 6학년 과학 동아리 학생 수

---

### 2단계: 해결 방법 계획  
- 각 학년별 학생 수를 구한다.  
- 각 학년별 과학 동아리 학생 수를 구한다.  
- 두 학년의 과학 동아리 학생 수를 합산한다.

---

### 3단계: 계획 실행  
- 5학년 학생 수 = 500 × 0.30 = 150명  
- 6학년 학생 수 = 500 × 0.20 = 100명  

- 5학년 과학 동아리 학생 수 = 150 × (1 - 0.60) = 150 × 0.40 = 60명  
- 6학년 과학 동아리 학생 수 = 100 × (1 - 0.70) = 100 × 0.30 = 30명  

- 과학 동아리 학생 수 = 60 + 30 = 90명

---

### 4단계: 검토  
- 계산 과정이 논리적이고 정확하다.  
- 다른 방법으로는 전체 학생 수에서 수학 동아리 학생 수를 빼는 방법도 있으나, 위 방법이 더 직관적이다.

---

### 답안:  
과학 동아리에는 **90명**의 학생이 있습니다.


In [25]:
# Ollama Phi3:mini 모델로 답안 생성

cot_chain = cot_prompt | ollama

answer = cot_chain.invoke({"question": question})

print(answer.content)

1. 학생들의 수를 기준으로 5학년이 있는 학생의 수를 구합니다: 500 * 30% = 150명
2. 6학년 학생의 수를 구합니다: 500 * 20% = 100명
3. 5학년 학생들의 수로서 수학 동아리에 있는 학생의 수를 계산합니다: 150 * 60% = 90명
4. 6학년 학생들의 수로서 수학 동아리에 있는 학생의 수를 계산합니다: 100 * 70% = 70명
5. 과학 동아리에 있는 학생들의 수를 구합니다: (90 + 70) - 500 * 30% * 40% (수학 동아리 전원이 모든 과학 동아리에 포함되지 않음) = 160 - 60 = 100명
6. 5학년 학생들의 수로서 과학 동아리에 있는 학생의 수를 계산합니다: 150 * 40% = 60명
7. 6학년 학생들의 수로서 과학 동아리에 있는 학생의 수를 계산합니다: 100 * 30% = 30명
8. 모든 학생들이 동원을 통해 과학 동아리에 포함되지 않는다면: 60 + 30 - 500 * 20% * 10% (6학년 전원이 모든 과학 동아리에 포함되지 않음) = 90 - 10 = 80명

따라서, 과학 동아리에는 80명의 학생이 있습니다.


In [29]:
# 다른 오픈소스 모델로 테스트
from langchain_ollama import ChatOllama

gemma = ChatOllama(
    model='gemma2:2b',
    temperature=0.3,  # 응답의 무작위성 조절 (0: 결정적, 1: 창의적)
    top_p=0.9,        # 누적 확률 기반 토큰 선택
)

deepseek = ChatOllama(
    model='deepseek-r1:7b',
    temperature=0.3,  # 응답의 무작위성 조절 (0: 결정적, 1: 창의적)
    top_p=0.9,        # 누적 확률 기반 토큰 선택
)

In [30]:
# GEMMA2 모델로 답안 생성

cot_chain = cot_prompt | gemma

answer = cot_chain.invoke({"question": question})

print(answer.content)

## 문제 해결 과정: 과학 동아리 학생 수

**1. 문제 이해하기:**

* **주어진 정보:** 학교에 500명의 학생이 있습니다. 
    * 30%는 5학년, 20%는 6학년입니다.
    * 5학년 중 60%는 수학 동아리, 나머지는 과학 동아리에 속합니다.
    * 6학년 중 70%는 수학 동아리, 나머지는 과학 동아리에 속합니다.

**2. 해결 방법 계획:**

* **단계별 분석:**  5학년과 6학년 학생들의 동아리 구성을 파악해야 합니다.
    * 각 단계별로 필요한 정보를 찾는 전략을 세우세요. (예: 숫자 연산, 비율 계산 등)

**3. 계획 실행:**

* **5학년 동아리 분석:**  
    1. 5학년 학생 수 = 500명 * 30% = 150명
    2. 60%는 수학 동아리, 40%는 과학 동아리: 150명 * 0.6 = 90명 (수학 동아리) + 150명 * 0.4 = 60명 (과학 동아리)
* **6학년 동아리 분석:**  
    1. 6학년 학생 수 = 500명 * 20% = 100명
    2. 70%는 수학 동아리, 30%는 과학 동아리: 100명 * 0.7 = 70명 (수학 동아리) + 100명 * 0.3 = 30명 (과학 동아리)

**4. 검토:**

* **답안 확인:**  
    * 과학 동아리 학생 수: 60명 (5학년 수학 동아리) + 30명 (6학년 과학 동아리) = 90명


**결론:** 학교에서 과학 동아리에는 **90명**의 학생이 있습니다. 






In [31]:
# Ollama DeepSeek 추론 모델로 답안 생성 (답변을 생성하기 위해 생각을 먼저 하는 모델로 zero-shot 적용)

zero_shot_chain = zero_shot_prompt | deepseek

answer = zero_shot_chain.invoke({"question": question})

print(answer.content)



**문제:**
학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
6학년 학생들 중 70%는 수학 동아리에 있고, 나머지
 science 동아리에 있습니다.
science 동아리에는 몇 명의 학생이 있나요?

**풀이:**

1. **5학년 학생들의 수를 구합니다:**
   \[
   500 \times 0.3 = 150 \text{명}
   \]

2. **6학년 학생들의 수를 구합니다:**
   \[
   500 \times 0.2 = 100 \text{명}
   \]

3. **5학년 학생 중 science 동아리에 속한 학생의 수를 구합니다:**
   \[
   150 \times 0.4 = 60 \text{명}
   \]
   ( science 동아리에 속하지 않기 때문에 100% - 60% = 40%)

4. **6학년 학생 중 science 동아리에 속한 학생의 수를 구합니다:**
   \[
   100 \times 0.3 = 30 \text{명}
   \]
   ( science동아리에 속하지 않기 때문에 100% - 70% = 30%)

5. **science 동아리에 속한 학생의 총 수를 구합니다:**
   \[
   60 + 30 = 90 \text{명}
   \]

**笞안:**
\[
\boxed{90}
\]


---

### **[실습]** 다음 논리적 추론 문제를 gemma2:2b 모델을 사용하여 3가지 유형의 프롬프트를 작성하여 해결하고, 그 성능을 비교합니다. 

(정답: 7번 이동)

In [32]:
question = """
농부가 늑대, 양, 양배추를 데리고 강을 건너야 함

제약조건:
1. 농부가 없을 때 늑대와 양이 같이 있으면 늑대가 양을 잡아먹음
2. 농부가 없을 때 양과 양배추가 같이 있으면 양이 양배추를 먹어버림
3. 보트에는 농부와 한 물건만 실을 수 있음

모두 안전하게 건너는데 몇 번 이동이 필요할까요. 숫자로 답변하세요.
"""

In [34]:
# [실습 1-1] Zero-shot 방식으로 답안 생성
# TODO: zero_shot_template을 참고하여 프롬프트 작성
# TODO: ollama 모델(gemma2:2b)과 체인을 구성
# TODO: question 변수로 invoke하여 답안 출력
# 힌트: 앞의 코드 구조를 참고하세요

# 여기에 코드를 작성하세요
zero_shot_template = """
다음 문제를 해결하시오:

문제: {question}

답안:
"""

zero_shot_prompt = PromptTemplate(
    input_variables=["question"],
    template=zero_shot_template
)

zero_gemma_chain = zero_shot_prompt | gemma

In [41]:
gemma_result = zero_gemma_chain.invoke({"question" : question})

print(gemma_result.content)

이 문제는 다양한 방법으로 해결할 수 있습니다.  다음은 몇 가지 가능한 방법과 그 이유입니다.

**1번 방법:** (3번 이동)

* **단계 1:** 농부, 양, 양배추를 함께 보트에 올려 강을 건너세요.
* **단계 2:** 농부는 양배추와 함께 보트에서 나가서 늑대와 함께 다른 곳으로 이동하세요.
* **단계 3:** 늑대를 제외하고 양과 양배추만 보트에 올려 강을 건너세요.

**2번 방법:** (5번 이동)

* **단계 1:** 농부, 양, 양배추를 함께 보트에 올려 강을 건너세요.
* **단계 2:** 농부는 양과 양배추만 보트에서 나가서 늑대와 함께 다른 곳으로 이동하세요.
* **단계 3:** 늑대를 제외하고 양과 양배추만 보트에 올려 강을 건너세요.
* **단계 4:** 농부는 양배추와 함께 보트에서 나가서 늑대와 함께 다른 곳으로 이동하세요.
* **단계 5:** 농부, 양, 양배추를 함께 보트에 올려 강을 건너세요.

**3번 방법:** (7번 이동)

* **단계 1:** 농부, 양, 양배추를 함께 보트에 올려 강을 건너세요.
* **단계 2:** 농부는 양과 양배추만 보트에서 나가서 늑대와 함께 다른 곳으로 이동하세요.
* **단계 3:** 늑대를 제외하고 양과 양배추만 보트에 올려 강을 건너세요.
* **단계 4:** 농부는 양배추와 함께 보트에서 나가서 늑대와 함께 다른 곳으로 이동하세요.
* **단계 5:** 농부, 양, 양배추를 함께 보트에 올려 강을 건너세요.
* **단계 6:** 농부는 양과 양배추만 보트에서 나가서 늑대와 함께 다른 곳으로 이동하세요.
* **단계 7:** 농부, 양, 양배추를 함께 보트에 올려 강을 건너세요.







In [39]:
# [실습 1-2] One-shot 방식으로 답안 생성
# TODO: one_shot_template을 참고하여 예시를 포함한 프롬프트 작성
# 힌트: 늑대-양-양배추 문제와 유사한 간단한 예시를 작성해보세요

# 여기에 코드를 작성하세요

one_shot_template = """
다음 문제를 해결하시오:

예시문제: {example}
예시답안: {example_response}

문제: {question}

답안:

"""

example = """
4명의 사람(A, B, C, D)이 밤에 손전등 하나를 가지고 외나무다리를 건너야 합니다.
제약조건:
1. 다리는 한 번에 최대 2명까지만 동시에 건널 수 있습니다.
2. 밤이라 어둡기 때문에 다리를 건널 때는 반드시 손전등을 지니고 가야 합니다. (즉, 다리를 건너간 후 반대편으로 손전등을 가져다주기 위해 누군가는 다시 돌아와야 합니다.)
3. 두 사람이 함께 다리를 건널 때는 더 느린 사람의 속도에 맞춰서 걸어야 합니다.
각 사람이 다리를 건너는 데 걸리는 시간은 다음과 같습니다:
- A: 1분
- B: 2분
- C: 5분
- D: 10분
네 사람 모두가 안전하게 다리를 건너는 데 필요한 최소 시간은 몇 분일까요? 숫자로 답변하세요.
"""

example_response = """
### 1. A와 B가 건너기 (2분)
### 2. A 복귀 (1분)
### 3. C와 D 건너기 (10분)
### 4. B 복귀 (2분)
### 5. A와 B 건너기 (2분)

답은 총 17 분 입니다.
"""

one_shot_prompt = PromptTemplate(
   input_variables=["example", "example_response", "question"],
   template=one_shot_template
)

gemma_one_shot_chain = one_shot_prompt | gemma




In [40]:
gemma_one_shot_result = gemma_one_shot_chain.invoke({"question" : question, "example" : example, "example_response" : example_response})


pprint(gemma_result)

AIMessage(content='This is a classic problem often referred to as the "Wolf, Sheep, and Cabbage" puzzle! Here\'s how to solve it:\n\n**Steps:**\n\n1. **Take the sheep across:**  Since the wolf will eat the sheep if they are together, take the sheep across first. \n2. **Return with the cabbage:** Leave the sheep behind and bring the cabbage across alone.\n3. **Bring back the sheep:** Now that the wolf is not present, bring the sheep across.\n4. **Take the wolf across:**  The wolf can now be safely taken across with the sheep. \n\n\n**Answer: 2 moves** \n\n\n\nLet me know if you\'d like to try another logic puzzle! 😊 \n', additional_kwargs={}, response_metadata={'model': 'gemma2:2b', 'created_at': '2026-06-17T13:11:01.6131583Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2709602700, 'load_duration': 337276800, 'prompt_eval_count': 164, 'prompt_eval_duration': 40660000, 'eval_count': 154, 'eval_duration': 2277116000, 'logprobs': None, 'model_name': 'gemma2:2b', 'model_provider

In [ ]:
# [실습 1-3] CoT 방식으로 답안 생성
# TODO: 단계별 추론 과정을 명시하는 프롬프트 작성
# 검증: 최종 답이 7번 이동인지 확인하세요

# 여기에 코드를 작성하세요

---

## **Self-Consistency**

* Self-Consistency는 AI 모델에게 하나의 문제에 대해 다양한 접근 방식으로 해결하도록 요청하는 기법으로, 여러 경로를 통해 도출된 결과들의 일관성을 확인함으로써 답변의 신뢰성을 높입니다.

* 이 방법은 특히 수학 문제나 논리적 추론이 필요한 과제에서 효과적이며, 서로 다른 방법으로 도출된 결과가 일치하는지 검증함으로써 오류 가능성을 최소화할 수 있습니다.

* Self-Consistency의 장점은 답변의 정확성을 높일 수 있다는 것이지만, 여러 번의 계산과 추론이 필요하므로 처리 시간이 길어지고 컴퓨팅 리소스 사용량이 증가한다는 단점도 존재합니다.

* 또한 이 기법은 Chain of Thought (CoT) 프롬프팅과 결합하여 사용할 경우 더욱 강력한 효과를 발휘할 수 있습니다.

- 논문: https://arxiv.org/abs/2203.11171


In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 프롬프트 템플릿 생성
self_consistency_template = """
다음 문제를 세 가지 다른 방법으로 해결하시오:

문제: {question}

세 가지 풀이 방법:
1) 직접 계산 방법:
   - 주어진 숫자를 직접 계산

2) 비율 활용 방법:
   - 전체에 대한 비율로 계산

3) 단계별 분해 방법:
   - 문제를 작은 부분으로 나누어 계산

각 방법의 답안을 제시하고, 결과가 일치하는지 확인하시오.

답안:
"""

self_consistency_prompt = PromptTemplate(
   input_variables=["question"],
   template=self_consistency_template
)

# 테스트용 문제
question = """
학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
과학 동아리에는 몇 명의 학생이 있나요?
"""

llm = ChatOpenAI(
    model='gpt-4.1-mini',
    temperature=0.3,  # 응답의 무작위성 조절 (0: 결정적, 1: 창의적)
    top_p=0.9,        # 누적 확률 기반 토큰 선택
)

# OpenAI gpt-4.1-nano 모델로 답안 생성
self_consistency_chain = self_consistency_prompt | llm 
answer = self_consistency_chain.invoke({"question": question})

print(answer.content)

문제:  
학교에 학생 500명  
- 5학년: 30% → 500 × 0.3 = 150명  
- 6학년: 20% → 500 × 0.2 = 100명  
5학년 중  
- 수학 동아리: 60% → 150 × 0.6 = 90명  
- 과학 동아리: 40% → 150 × 0.4 = 60명  
6학년 중  
- 수학 동아리: 70% → 100 × 0.7 = 70명  
- 과학 동아리: 30% → 100 × 0.3 = 30명  

과학 동아리 학생 수 = 5학년 과학 + 6학년 과학 = 60 + 30 = 90명

---

### 1) 직접 계산 방법

- 5학년 학생 수: 500 × 0.3 = 150명  
- 6학년 학생 수: 500 × 0.2 = 100명  
- 5학년 과학 동아리: 150 × (1 - 0.6) = 150 × 0.4 = 60명  
- 6학년 과학 동아리: 100 × (1 - 0.7) = 100 × 0.3 = 30명  
- 과학 동아리 총 인원: 60 + 30 = **90명**

---

### 2) 비율 활용 방법

전체 학생 500명 중 과학 동아리 비율을 구함

- 5학년 과학 동아리 비율 = 0.3 × (1 - 0.6) = 0.3 × 0.4 = 0.12  
- 6학년 과학 동아리 비율 = 0.2 × (1 - 0.7) = 0.2 × 0.3 = 0.06  
- 전체 과학 동아리 비율 = 0.12 + 0.06 = 0.18

따라서 과학 동아리 학생 수 = 500 × 0.18 = **90명**

---

### 3) 단계별 분해 방법

1) 5학년 학생 수 구하기  
   - 500 × 0.3 = 150명

2) 5학년 과학 동아리 학생 수 구하기  
   - 150명 중 40% → 150 × 0.4 = 60명

3) 6학년 학생 수 구하기  
   - 500 × 0.2 = 100명

4) 6학년 과학 동아리 학생 수 구하기  
   - 100명 중 30% → 100 × 0.3 = 30명

5) 과학 동아리 총 인원  
   - 60 + 30 = **90

---

## **Program-Aided Language (PAL)**

* PAL은 자연어 문제를 프로그래밍적 사고방식으로 접근하도록 하는 기법으로, 복잡한 문제를 코드나 의사코드 형태로 분해하여 해결하는 방식입니다. 이를 통해 문제 해결 과정을 더욱 구조화하고 체계적으로 만들 수 있습니다.

* 이 접근 방식의 큰 장점은 프로그래밍 언어의 정확성과 논리성을 활용하여 모호함을 줄이고, 각 단계를 명확하게 정의할 수 있다는 것입니다. 특히 수학적 계산, 데이터 처리, 알고리즘적 문제 해결에서 뛰어난 성능을 보입니다.

* PAL의 특징적인 점은 실제 실행 가능한 코드를 생성할 수 있다는 것으로, 이는 결과의 검증이 용이하고 필요한 경우 수정이나 최적화가 가능하다는 장점이 있습니다. 

- 논문: https://arxiv.org/pdf/2211.10435


In [5]:
from langchain_core.prompts import PromptTemplate

# 프롬프트 템플릿 생성
pal_template = """
다음 문제를 Python 프로그래밍 방식으로 해결하시오:

문제: {question}

# 문제 해결을 위한 Python 스타일 의사코드:
def solve_problem():
    # 1. 변수 정의
    # - 주어진 값들을 변수로 저장
    
    # 2. 계산 과정
    # - 필요한 계산을 단계별로 수행
    
    # 3. 결과 반환
    # - 최종 결과 계산 및 반환
    
답안:
"""

pal_prompt = PromptTemplate(
    input_variables=["question"],
    template=pal_template
)

# 테스트용 문제
question = """
학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
과학 동아리에는 몇 명의 학생이 있나요?
"""

# OpenAI gpt-4.1-nano 모델로 답안 생성
pal_chain = pal_prompt | llm 
answer = pal_chain.invoke({"question": question})

print(answer.content)

```python
def solve_problem():
    # 1. 변수 정의
    total_students = 500
    percent_5th = 0.30
    percent_6th = 0.20

    percent_math_5th = 0.60
    percent_science_5th = 1 - percent_math_5th

    percent_math_6th = 0.70
    percent_science_6th = 1 - percent_math_6th

    # 2. 계산 과정
    num_5th = total_students * percent_5th
    num_6th = total_students * percent_6th

    science_5th = num_5th * percent_science_5th
    science_6th = num_6th * percent_science_6th

    total_science = science_5th + science_6th

    # 3. 결과 반환
    return int(total_science)

# 결과 출력
print(solve_problem())
```

---

**설명:**  
- 전체 학생 수 500명 중 5학년과 6학년 학생 수를 각각 구합니다.  
- 각 학년별로 수학 동아리와 과학 동아리 비율을 적용해 과학 동아리 학생 수를 구합니다.  
- 5학년과 6학년 과학 동아리 학생 수를 합산하여 최종 결과를 반환합니다.


---

## **Reflexion**

* Reflexion은 AI가 자신의 이전 답변을 스스로 검토하고 평가하여 개선하는 메타인지적 프롬프팅 기법으로, 이를 통해 응답의 질을 점진적으로 향상시킬 수 있습니다.

* 이 방법은 AI가 자신의 답변에서 부족한 점, 오류, 또는 개선이 필요한 부분을 스스로 찾아내고 수정하도록 함으로써, 더 정확하고 완성도 높은 답변을 도출할 수 있게 합니다. 특히 복잡한 분석이나 창의적인 작업에서 효과적입니다.

* Reflexion의 강점은 AI가 자기 평가를 통해 지속적으로 개선된 결과물을 제공할 수 있다는 것이지만, 여러 번의 반복적인 검토와 수정 과정이 필요하므로 시간과 컴퓨팅 자원이 더 많이 소요될 수 있다는 제한점이 있습니다.

* 이 기법은 특히 글쓰기, 코드 리뷰, 분석 리포트 작성 등 높은 품질의 출력이 요구되는 작업에서 매우 유용하게 활용될 수 있습니다.

- 논문: https://arxiv.org/abs/2303.11366

In [6]:
from langchain_core.prompts import PromptTemplate

# 프롬프트 템플릿 생성
reflexion_template = """
다음 문제에 대해 단계적으로 해결하여 초기 답안을 작성하고, 자체 평가 후 개선하시오:

문제: {question}

1단계: 초기 답안
---
[여기에 첫 번째 답안 작성]

2단계: 자체 평가
---
- 정확성 검토
- 논리적 오류 확인
- 설명의 명확성 평가
- 개선이 필요한 부분 식별

3단계: 개선된 답안
---
[평가를 바탕으로 개선된 답안 작성]

답안:
"""

reflexion_prompt = PromptTemplate(
    input_variables=["question"],
    template=reflexion_template
)

# 테스트용 문제
question = """
학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
과학 동아리에는 몇 명의 학생이 있나요?
"""

# OpenAI gpt-4.1-nano 모델로 답안 생성
reflexion_chain = reflexion_prompt | llm  # 이미 정의된 llm 재사용
answer = reflexion_chain.invoke({"question": question})

print(answer.content)

1단계: 초기 답안
---
주어진 정보를 정리해보자.

- 전체 학생 수: 500명
- 5학년 학생 비율: 30% → 500 × 0.30 = 150명
- 6학년 학생 비율: 20% → 500 × 0.20 = 100명

5학년 학생 중:
- 수학 동아리: 60% → 150 × 0.60 = 90명
- 과학 동아리: 나머지 40% → 150 × 0.40 = 60명

6학년 학생 중:
- 수학 동아리: 70% → 100 × 0.70 = 70명
- 과학 동아리: 나머지 30% → 100 × 0.30 = 30명

과학 동아리 학생 수 = 5학년 과학 동아리 + 6학년 과학 동아리 = 60 + 30 = 90명

답: 과학 동아리에는 90명의 학생이 있습니다.

---

2단계: 자체 평가
---
- 정확성 검토: 계산 과정과 비율 적용이 올바르게 이루어졌다.
- 논리적 오류 확인: 5학년과 6학년 학생 수를 정확히 구하고, 동아리 비율을 올바르게 적용하였다.
- 설명의 명확성 평가: 각 단계별 계산 과정을 명확히 제시하여 이해하기 쉽다.
- 개선이 필요한 부분 식별: 전체 학생 중 5학년과 6학년 외의 학생(50%)에 대한 언급이 없는데, 문제에서 과학 동아리 인원만 묻고 있으므로 무시해도 무방하다. 다만, 전체 학생 중 과학 동아리 인원을 묻는 것인지, 5학년과 6학년만 포함하는지 명확히 할 필요가 있다. 문제 문맥상 5학년과 6학년만 동아리 활동을 하는 것으로 보인다.

---

3단계: 개선된 답안
---
학교에는 총 500명의 학생이 있습니다. 이 중 30%인 150명은 5학년이고, 20%인 100명은 6학년입니다.

5학년 학생 중 60%는 수학 동아리에, 나머지 40%는 과학 동아리에 속해 있습니다. 따라서 5학년 과학 동아리 학생 수는 150 × 0.40 = 60명입니다.

6학년 학생 중 70%는 수학 동아리에, 나머지 30%는 과학 동아리에 속해 있습니다. 따라서 6학년 과학 동아리 학생 수는 100 × 0.30 = 30명입니다.

따라서 과학 동아리 학생

### **[실습 2]** [실습 1]의 논리적 추론 문제를 gemma2:2b 모델을 사용하여 3가지 유형의 프롬프트를 작성하여 해결하고, 그 성능을 비교합니다. 

(정답: 7번 이동)

In [8]:
# [실습 2-1] Self-Consistency 방식으로 답안 생성
# TODO: self_consistency_template을 참고하여 프롬프트 작성
# TODO: gemma2:2b 모델과 체인을 구성 (gemma 변수 사용)
# TODO: question 변수로 invoke하여 답안 출력
# 힌트: 앞의 코드 구조를 참고하세요

# 여기에 코드를 작성하세요

from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama


# 프롬프트 템플릿 생성
self_consistency_template = """
다음 문제를 세 가지 다른 방법으로 해결하시오:

문제: {question}

세 가지 풀이 방법:
1) 직접 계산 방법:
   - 주어진 숫자를 직접 계산

2) 비율 활용 방법:
   - 전체에 대한 비율로 계산

3) 단계별 분해 방법:
   - 문제를 작은 부분으로 나누어 계산

각 방법의 답안을 제시하고, 결과가 일치하는지 확인하시오.

답안:
"""

self_consistency_prompt = PromptTemplate(
   input_variables=["question"],
   template=self_consistency_template
)

# 테스트용 문제
question = """
학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
과학 동아리에는 몇 명의 학생이 있나요?
"""

gemma = ChatOllama(
    model='gemma2:2b',
    temperature=0.3,  # 응답의 무작위성 조절 (0: 결정적, 1: 창의적)
    top_p=0.9,        # 누적 확률 기반 토큰 선택
)

self_consistency_chain = self_consistency_prompt | gemma 
answer = self_consistency_chain.invoke({"question": question})

print(answer.content)

## 학교 학생 동아리 수학 문제 해결 방법 세 가지

**1) 직접 계산 방법:**

* **5학년 학생:** 500명 * 30% = 150명
* **6학년 학생:** 500명 * 20% = 100명
* **수학 동아리 5학년:** 150명 * 60% = 90명
* **과학 동아리 5학년:** 150명 * 40% = 60명 
* **6학년 수학 동아리:** 100명 * 70% = 70명
* **과학 동아리 6학년:** 100명 * 30% = 30명

**결론:** 과학 동아리에는 30명의 학생이 있습니다.


**2) 비율 활용 방법:**

* **5학년:** 전체 학생의 30% (500명 * 30%)
* **6학년:** 전체 학생의 20% (500명 * 20%)
* **수학 동아리 5학년:** 5학년의 60% (150명 * 60%)
* **과학 동아리 5학년:** 5학년의 나머지 (150명 * 40%)
* **수학 동아리 6학년:** 6학년의 70% (100명 * 70%)
* **과학 동아리 6학년:** 6학년의 나머지 (100명 * 30%)

**결론:** 과학 동아리에는 30명의 학생이 있습니다.


**3) 단계별 분해 방법:**

1. **5학년 학생:** 전체 학생의 30% (500명 * 30%)
2. **6학년 학생:** 전체 학생의 20% (500명 * 20%)
3. **수학 동아리 5학년:** 5학년의 60% (150명 * 60%)
4. **과학 동아리 5학년:** 5학년의 나머지 (150명 * 40%)
5. **수학 동아리 6학년:** 6학년의 70% (100명 * 70%)
6. **과학 동아리 6학년:** 6학년의 나머지 (100명 * 30%)

**결론:** 과학 동아리에는 30명의 학생이 있습니다.



## 결과 비교 및 분석

* 세 가지 방법 모두 **과학 동아리에 30명의 학생이 있다는 결론을 도출합니다.** 
* 직접 계산, 비율 활용, 단계별 분해 방법은 각각 다른 방식으로 문제를 해결하는 것을 보여줍니다.





In [9]:
# [실습 2-2] Program-Aided Language 방식으로 답안 생성
# TODO: pal_template을 참고하여 프로그래밍적 프롬프트 작성
# TODO: gemma2:2b 모델과 체인을 구성
# 힌트: Python 의사코드 형태로 문제 해결 과정을 구조화하세요

# 여기에 코드를 작성하세요

from langchain_core.prompts import PromptTemplate

# 프롬프트 템플릿 생성
pal_template = """
다음 문제를 Python 프로그래밍 방식으로 해결하시오:

문제: {question}

# 문제 해결을 위한 Python 스타일 의사코드:
def solve_problem():
    # 1. 변수 정의
    # - 주어진 값들을 변수로 저장
    
    # 2. 계산 과정
    # - 필요한 계산을 단계별로 수행
    
    # 3. 결과 반환
    # - 최종 결과 계산 및 반환
    
답안:
"""

pal_prompt = PromptTemplate(
    input_variables=["question"],
    template=pal_template
)

# 테스트용 문제
question = """
학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
과학 동아리에는 몇 명의 학생이 있나요?
"""

pal_chain = pal_prompt | gemma 
answer = pal_chain.invoke({"question": question})

print(answer.content)

```python
def solve_problem():
  """
  학교 학생들의 동아리 참여율에 대한 정보를 사용하여 과학 동아리의 수학 학생 수를 계산하는 함수.

  Returns:
    과학 동아리의 학생 수 (int)
  """

  # 1. 변수 정의
  total_students = 500  # 학교 전체 학생 수
  fifth_graders = total_students * 0.3  # 5학년 학생 수
  sixth_graders = total_students * 0.2  # 6학년 학생 수

  # 2. 계산 과정
  math_club_fifth = fifth_graders * 0.6  # 5학년 수학 동아리 참여율 (60%)
  science_club_fifth = fifth_graders * 0.4  # 5학년 과학 동아리 참여율 (40%)

  math_club_sixth = sixth_graders * 0.7  # 6학년 수학 동아리 참여율 (70%)
  science_club_sixth = sixth_graders * 0.3  # 6학년 과학 동아리 참여율 (30%)

  # 3. 결과 반환
  science_club_students = math_club_sixth + science_club_sixth # 과학 동아리 학생 수 계산

  return science_club_students


if __name__ == "__main__":
  result = solve_problem()
  print(f"과학 동아리에는 {result} 명의 학생이 있습니다.") 
```



**설명:**

1. **변수 정의**: 문제에서 주어진 정보들을 변수에 저장합니다 (예: `total_students`, `fifth_graders`, `sixth_graders`).
2. **계산 과정**: 각 변수를 이용하여 필요한 계산을 수행합니다. 
    *  5학년, 6학년 학생들의 수를 계산하고, 그 중 수학 동아리 참여율도 계산합니다.
3. *

In [10]:
# [실습 2-3] Reflexion 방식으로 답안 생성
# TODO: reflexion_template을 참고하여 자체 평가 프롬프트 작성
# TODO: gemma2:2b 모델과 체인을 구성
# 검증: 초기 답안과 개선된 답안의 차이를 비교하세요

# 여기에 코드를 작성하세요

from langchain_core.prompts import PromptTemplate

# 프롬프트 템플릿 생성
reflexion_template = """
다음 문제에 대해 단계적으로 해결하여 초기 답안을 작성하고, 자체 평가 후 개선하시오:

문제: {question}

1단계: 초기 답안
---
[여기에 첫 번째 답안 작성]

2단계: 자체 평가
---
- 정확성 검토
- 논리적 오류 확인
- 설명의 명확성 평가
- 개선이 필요한 부분 식별

3단계: 개선된 답안
---
[평가를 바탕으로 개선된 답안 작성]

답안:
"""

reflexion_prompt = PromptTemplate(
    input_variables=["question"],
    template=reflexion_template
)

# 테스트용 문제
question = """
학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
과학 동아리에는 몇 명의 학생이 있나요?
"""

reflexion_chain = reflexion_prompt | gemma  
answer = reflexion_chain.invoke({"question": question})

print(answer.content)

## 문제 해결 과정

**1 단계: 초기 답안**

500명의 학생 중 30%는 5학년이므로, 5학년 학생 수는 500 * 0.3 = 150 명입니다. 
6학년 학생 수는 500 * 0.2 = 100명입니다.

5학년 중 60%가 수학 동아리에 있으므로, 수학 동아리 학생 수는 150 * 0.6 = 90명입니다.
나머지는 과학 동아리에 있으므로, 과학 동아리 학생 수는 150 - 90 = 60명입니다.

**2 단계: 자체 평가**

* **정확성 검토:** 초기 답안은 정확한 계산을 기반으로 하였습니다.
* **논리적 오류 확인:**  모든 계산 과정에서 논리적 오류를 찾지 못했습니다.
* **설명의 명확성 평가:** 설명은 단순하고 명확하게 작성되었지만, 좀 더 자세한 정보를 추가하여 이해도를 높일 수 있습니다. 
* **개선이 필요한 부분 식별:**  더 많은 정보를 제공하면서 문제 해결 과정을 더욱 명확하게 보여줄 수 있습니다.


**3 단계: 개선된 답안**

500명의 학생 중 30%는 5학년이므로, 5학년 학생 수는 500 * 0.3 = 150 명입니다. 
6학년 학생 수는 500 * 0.2 = 100명입니다.

5학년 학생 중 60%가 수학 동아리에 있으므로, 수학 동아리 학생 수는 150 * 0.6 = 90명입니다.
나머지는 과학 동아리에 있으므로, 과학 동아리 학생 수는 150 - 90 = 60명입니다.

**결론:**  문제 해결 과정을 단계별로 설명하고, 정확성과 논리적 오류를 확인하여 개선된 답안을 제시했습니다. 



